## Client


In [1]:
from intuitlib.client import AuthClient

from intuit_cdc.config import SETTINGS

auth_client = AuthClient(
    SETTINGS.client_id,
    SETTINGS.client_secret,
    SETTINGS.redirect_uri,
    SETTINGS.environment,
)

In [2]:
from quickbooks import QuickBooks

# Create the QuickBooks client instance
qb_client = QuickBooks(
    auth_client=auth_client,
    refresh_token="AB11749944529gUp0qc6NssKrG0oJ90ro9Y6C8JLj1CntXhus5",
    company_id="9341454218315322",
)

## Create a dummy invoice


In [3]:
from quickbooks.objects.customer import Customer

# Query customer by email
# customer_email = "customer@example.com"
query = "SELECT * FROM Customer"
customers = Customer.query(query, qb=qb_client)

# Print customer IDs and names
# for customer in customers:
#     print(f"Customer ID: {customer.Id}, Name: {customer.DisplayName}")

In [4]:
from quickbooks.objects.item import Item

# Query item by SKU
query = "SELECT * FROM Item"
items = Item.query(query, qb=qb_client)

# Print item IDs and names
# for item in items:
#     print(f"Item ID: {item.Id}, Name: {item.Name}")

In [5]:
from quickbooks.objects.customer import Customer
from quickbooks.objects.item import Item
from quickbooks.objects.invoice import Invoice
from quickbooks.objects.detailline import SalesItemLine, SalesItemLineDetail

# Retrieve an existing customer and item (replace with valid IDs from your QBO account)
customer = Customer.get("3", qb=qb_client)
item = Item.get("3", qb=qb_client)

# Create a line item for the invoice
line = SalesItemLine()
line.Amount = 150.00
line.SalesItemLineDetail = SalesItemLineDetail()
line.SalesItemLineDetail.ItemRef = item.to_ref()

# Create the invoice and assign the customer and line item
invoice = Invoice()
invoice.CustomerRef = customer.to_ref()
invoice.Line = [line]

# Save the invoice in QBO
invoice.save(qb=qb_client)

print("Invoice created with ID:", invoice.Id)

Invoice created with ID: 146


In [6]:
from quickbooks.objects.bill import Bill
from quickbooks.objects.invoice import Invoice
from quickbooks.objects.journalentry import JournalEntry

entities = [
    Invoice,
    Bill,
    JournalEntry,
]

In [7]:
from datetime import datetime

changed_since = datetime(2025, 3, 1, 0, 0, 0)  # Adjust as needed

In [8]:
# Poll CDC for invoices and journal entries
from quickbooks.cdc import change_data_capture

cdc_response = change_data_capture(
    qbo_class_list=[Invoice],
    timestamp=changed_since,
    qb=qb_client,
)

In [9]:
for invoice in cdc_response.Invoice:
    print(f"Detected invoice change: {invoice}")

Detected invoice change: 150.0
Detected invoice change: 150.0
